# IN4050 Assignment 1: Travelling Salesman problem

# Levente Orha - leventeo@uio.no

## Introduction, Problem

This notebook attempts to solve an instance of the traveling salesman problem (TSP) using different methods. The goal is to become familiar with evolutionary algorithms and to  appreciate their effectiveness on a difficult search problem.


|  &nbsp;   | Barcelona | Belgrade |  Berlin | Brussels | Bucharest | Budapest |
|:---------:|:---------:|:--------:|:-------:|:--------:|:---------:|:--------:|
| Barcelona |     0     |  1528.13 | 1497.61 |  1062.89 |  1968.42  |  1498.79 |
|  Belgrade |  1528.13  |     0    |  999.25 |  1372.59 |   447.34  |  316.41  |
|   Berlin  |  1497.61  |  999.25  |    0    |  651.62  |  1293.40  |  1293.40 |
|  Brussels |  1062.89  |  1372.59 |  651.62 |     0    |  1769.69  |  1131.52 |
| Bucharest |  1968.42  |  447.34  | 1293.40 |  1769.69 |     0     |  639.77  |
|  Budapest |  1498.79  |  316.41  | 1293.40 |  1131.52 |   639.77  |     0    |


<center>Figure 1: First 6 cities from csv file.</center>

### Problem

The traveling salesman, wishing to disturb the residents of the major cities in some region of the world in the shortest time possible, is faced with the problem of finding the shortest tour among the cities. A tour is a path that starts in one city, visits all of the other cities, and then returns to the starting point. The relevant pieces of information, then, are the cities and the distances between them. In this instance of the TSP, a number of European cities are to be visited. Their relative distances are given in the data file, *european_cities.csv*.

## Imports, setup

### Helper codes for visualization, time measurements and others

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from TSP import TSP # Import the Travelling Salesperson code
tsp = TSP("datafiles/european_cities.csv", "datafiles/map.png") # Create an instance
city_names = list(tsp.city_coords.keys()) # Getting the cities in alphabetical order

### Algorithm classes imports

In [ ]:
from Algorithms.exhaustive_search import ExhaustiveSearch
from Algorithms.hill_climb import HillClimb
from Algorithms.genetic_algorithm import GeneticAlgorithm

## Exhaustive search

In [ ]:
# Create the instance of the algorithm
exhSearch = ExhaustiveSearch(tsp.tour_length, "min")

### 6 cities

In [ ]:
# Run for 6 cities
best_order, best_length = tsp.time_function(
    "6 cities",
    exhSearch.search,
    list(range(1,6)),
    starting_point=0)
print(f"Best order: {best_order} with length {best_length:.2f} km")
tsp.plot_plan([city_names[i] for i in best_order])

### 10 cities

In [ ]:
# Run for 10 cities
best_order, best_length = tsp.time_function(
    "10 cities",
    exhSearch.search,
    list(range(1,10)),
    starting_point=0)
print(f"Best order: {best_order} with length {best_length:.2f} km")
tsp.plot_plan([city_names[i] for i in best_order])

### Results

For 6 cities:
* Shortest route: 0-1-4-5-2-3
* Distance: 5018.81 km
* Elapsed time: 0.017 s

For 10 cities:
* Shortest route: 0-1-6-4-5-2-6-8-3-7
* Distance: 7486.31 km
* Elapsed time: 39.72 s

For 24 cities:

The elapsed time scales with the number of cities used, giving a scaling factor:
$$
C(n) = \frac{T(n-1)}{(n-1)!}
$$
which gives $C(10) = 1.24 \cdot 10^{-4}$ s and $C(6) = 1.75 \cdot 10^{-4}$ s. Based on this, $T(24) = 1.495 \cdot 10^{-4} \cdot 23! \approx 3.86 \cdot 10^{18} \text{s} \approx 1.22 \cdot 10^{11} $ years.

## Hill climbing

In [ ]:
hillClimb = HillClimb(tsp.tour_length, "min")

In [ ]:
def repeat_runs_hc(n_runs: int, n_cities: int, n_steps: int):
    """
    Run the Hill climb algorithm `n_runs` times from 0, track results
    """
    results = []
    for _ in range(n_runs):
        start_order = list(range(0, n_cities))
        np.random.shuffle(start_order) # One random permutation
        results.append(tuple(hillClimb.search(start_order, n_steps)))
    return results

### 10 cities

#### Timing

In [ ]:
# Create an ordering of cities and randomly shuffle it
start_order = list(range(0, 10))
np.random.shuffle(start_order)
_ = tsp.time_function("Hill climb for 10 cities", hillClimb.search, start_order, 200)

#### Repeated runs, with reporting results

In [ ]:
results = repeat_runs_hc(20, 10, 200)
best, worst = tsp.report_runs(results)

In [ ]:
tsp.plot_plan([city_names[i] for i in best["Order"]])

### 24 cities

#### Timing

In [ ]:
# Create an ordering of cities and randomly shuffle it
start_order = list(range(0, 24))
np.random.shuffle(start_order)
_ = tsp.time_function("Hill climb for 24 cities", hillClimb.search, start_order, 200)

#### Repeated runs, with reporting results

In [ ]:
results = repeat_runs_hc(20, 24, 200)
best, worst = tsp.report_runs(results)

In [ ]:
tsp.plot_plan([city_names[i] for i in best["Order"]])

## Genetic Algorithm

### Parameter, method selection

For this work, Partially Mapped Crossover has been selected as crossover method, Tournament selection for parent selection, and Random Swap for mutation.

In each run, regardless of the number of individuals and the number of cities, 50 generations had been run, as this gives enough space to evolve, while keeping computational costs low. For both number of cities, the algorithm seems to converge by the end (see figures).

The tournament size was choosen to be 3 to provide a moderate level of selection pressure. A larger value would prefer the overall better individuals, while this value gives chance to less fit individuals too.

In [ ]:
def repeat_ea(n_runs, population_size, n_cities, method = ""):
    results = []
    fitness_log = []
    for _ in range(n_runs):
        ga = GeneticAlgorithm(tsp.tour_length, 3, "min", method)
        population = ga.generate_population(n_cities, population_size)
        best_order, best_length = ga.run_EA(population, 50)
        results.append((best_order, best_length))
        fitness_log.append(ga.fitness_log)
    return results, np.mean(np.array(fitness_log), axis=0)

### 10 cities

#### Population size 20

In [ ]:
results20 = tsp.time_function("n_p = 20, n_c = 10", repeat_ea, 20, 20, 10)
_ = tsp.report_runs(results20[0])
tsp.plot_plan([city_names[i] for i in results20[0][-1][0]])

#### Population size 50

In [ ]:
results50 = tsp.time_function("n_p = 50, n_c = 10", repeat_ea, 20, 50, 10)
_ = tsp.report_runs(results50[0])
tsp.plot_plan([city_names[i] for i in results50[0][-1][0]])

#### Population size 100

In [ ]:
results100 = tsp.time_function("n_p = 100, n_c = 10", repeat_ea, 20, 100, 10)
_ = tsp.report_runs(results100[0])
tsp.plot_plan([city_names[i] for i in results100[0][-1][0]])

#### Fitness plot

In [ ]:
tsp.plot_fitness([results20[1], results50[1], results100[1]], ["20", "50", "100"])

### All cities

#### Population size 50

In [ ]:
results50 = tsp.time_function("n_p = 50, n_c = 24", repeat_ea, 20, 50, 24)
_ = tsp.report_runs(results50[0])
tsp.plot_plan([city_names[i] for i in results50[0][-1][0]])

#### Population size 100

In [ ]:
results100 = tsp.time_function("n_p = 100, n_c = 24", repeat_ea, 20, 100, 24)
_ = tsp.report_runs(results100[0])
tsp.plot_plan([city_names[i] for i in results100[0][-1][0]])

#### Population size 200

In [ ]:
results200 = tsp.time_function("n_p = 200, n_c = 24", repeat_ea, 20, 200, 24)
_ = tsp.report_runs(results200[0])
tsp.plot_plan([city_names[i] for i in results200[0][-1][0]])

#### Fitness plot

In [ ]:
tsp.plot_fitness([results50[1], results100[1], results200[1]], ["50", "100", "200"])

### Results

1. The Generic Algorithm found an even better tour with 10 cities than the exhaustive search, yielding only 7486.31 km. 
2. Runtimes
    * For 10 cities, GA performed with 20 s for a 100 individual in the population, while exhaustive search used 40 s.
    * For 24 cities, GA used only 27 s (100 population size), and not longer than the universe's life as exhaustive search.
3.  The total number of examined routes are given by
$$
(1 + n_{\text{generation}}) \cdot N_i = (1 + 50) \cdot 100 = 5\,100
$$
for both number of cities, as it only depends on the number of individuals ($N_i$) and the number of generations ($n_{\text{generation}}$). Exhaustive search checks all possible outcomes, which, for 10 cities is $(10-1)! = 9! = 362\,880$.

## Hybrid Algorithm

<span style="color: red">Note: </span>Algorithms implemented in the overall genetic algorithm code!

### Lamarckian run

In [ ]:
resultsL = tsp.time_function("Lamarckian run", repeat_ea, 20, 100, 24, method="L")
_ = tsp.report_runs(resultsL[0])

### Baldwinian run

In [ ]:
resultsB = tsp.time_function("Lamarckian run", repeat_ea, 20, 100, 24, method="B")
_ = tsp.report_runs(resultsB[0])

### Fitness

In [ ]:
tsp.plot_fitness([resultsL[1], resultsB[1]], ["Lamarckian", "Baldwinian"])

### Results

At each run, for each individual, an additional 20 evaluations are done, so the equation becomes
$$
(1 + n_{\text{generation}} \cdot 20) \cdot N_i = (1 + 1000) \cdot 100 = 100\,100
$$
evaluations done for both Lamarckian and Baldwinian runs.

However, both methods fail to find the shortest route in the given number of generations compared to the simple Genetic Algorithm, in a much longer time. Or, probably, it was me who implemented something wrong.